In [2]:
import os
import re
import json
import time
import pandas as pd
from pprint import pprint
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm
from pinecone import Pinecone, ServerlessSpec

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_CLOUD = os.getenv("PINECONE_CLOUD")      # aws
PINECONE_REGION = os.getenv("PINECONE_REGION")    # us-east-1
INDEX_NAME = os.getenv("INDEX_NAME")

pc = Pinecone(
    api_key=PINECONE_API_KEY
)

index = pc.Index(INDEX_NAME)

### NAIC

In [36]:
def embed_patent_text(text: str) -> list:
    
    response = client.embeddings.create(
        model="text-embedding-3-large",
        input=text
    )
    embedding = response.data[0].embedding

    assert len(embedding) == 3072
    return embedding

def retrieve_top_naics(
    query_vector: list,
    top_k: int = 15
) -> list:
    result = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )

    naics_candidates = []

    for match in result["matches"]:
        meta = match["metadata"]
        naics_candidates.append({
            "code": meta["NAICS_code"],      # 6자리
            "title": meta["NAICS_title"],
            "desc": meta["original_description"],
            "score": match["score"]
        })

    return naics_candidates

def build_naics_context_text(naics_candidates: list) -> str:
    naics_text = "\n".join(
        [
            f"- {str(c['code']).split('.')[0]}: {c['title']} — {c['desc']}"
            for c in naics_candidates
        ]
    )

    return naics_text


def extract_patent_metadata(text: str, naics_context: str):
    prompt = f"""
You are a patent classification and information extraction system.

You will be given:
1) Raw OCR text extracted from a patent publication.
2) A list of candidate NAICS industry codes (with descriptions).

The OCR text may contain OCR errors, duplicated lines, broken line breaks,
or reading-order issues. Do NOT attempt to fix or rewrite the text.

Your tasks are:
(A) Extract bibliographic and technical fields STRICTLY according to INID codes.
(B) Select the MOST APPROPRIATE NAICS industry code(s) from the GIVEN CANDIDATES ONLY.

----------------------------------------
NAICS SELECTION RULES (VERY IMPORTANT)
----------------------------------------
- Please note that the provided NAICS codes are already sorted in order of relevance, so take this into consideration.
- You MUST choose NAICS code(s) ONLY from the candidate list provided below.
- Select the code(s) that BEST match the patent's technical field and application.
- Base your decision primarily on:
  - IPC codes (INID 51)
  - Abstract (INID 57)
- If exactly one NAICS code is clearly the best fit, return ONLY one.
- If two or three codes are strongly relevant, you MAY return up to three.
- NEVER invent or infer NAICS codes not present in the candidate list.
- Return NAICS codes as a list of 6-digit strings.

----------------------------------------
IMPORTANT CONSTRAINTS
----------------------------------------
- Do NOT hallucinate missing information.
- Do NOT infer beyond the text.
- Output MUST be valid JSON.
- Output ONLY the JSON object.
- Return all extracted text fields (including title and abstract) in the ORIGINAL LANGUAGE of the patent as indicated by the country code.
- The "field" value MUST be assigned to EXACTLY ONE of the five allowed categories ("compu", "bio", "comm", "elec", "etc"); it MUST NOT be null under any circumstances.

----------------------------------------
OUTPUT FORMAT (JSON ONLY)
----------------------------------------
Return a JSON object with EXACTLY the following fields.
Use null if a field cannot be confidently extracted.

{{
  "country": "US",
  "title": "Example Title",
  "applicant_name": "Example Applicant",
  "applicant_number": "17/123,456",
  "applicant_date": "2021-02-22",
  "grant_number": null,
  "grant_date": null,
  "naics_code": ["325414"],
  "abstract": "Example abstract text...",
  "field": "bio",
  "ipc_info": [{{
    "code": "A61K 39/05",
    "short_description": "백신기술",
    "long_description": "면역 치료용 백신 조성물 관련 기술"
  }}]
}}

----------------------------------------
FIELD EXTRACTION RULES
----------------------------------------

1. country
- Infer from INID (19).
- If missing, infer from document kind or header in INID (12).
- Return ISO 2-letter country code (e.g., "US", "KR").
- Allowed exceptions: "PCT", "EP".
- If uncertain, return null.

2. title
- Extract ONLY from INID (54).
- Use the original text verbatim.

3. applicant_name
- Extract verbatim.
- Use:
  - INID (71) for U.S. documents.
  - INID (73) for Korean documents.

4. applicant_number
- Extract ONLY from INID (21).

5. applicant_date
- Extract ONLY from INID (22).
- Convert to ISO format YYYY-MM-DD.

6. grant_number
- Extract ONLY from INID (11).

7. grant_date
- Extract ONLY from INID (45).

8. ipc_code
- Extract ALL IPC codes under INID (51).
- Preserve original formatting.
- Return as a list.

9. abstract
- Extract FULL abstract from INID (57).
- Preserve verbatim text.

10. field
- Assign EXACTLY ONE of:
  "compu", "bio", "comm", "elec", "etc"
- Decide primarily from IPC codes.

11. ipc_long_description
- For EACH IPC code extracted under INID (51),
  provide a clear technical/industry-oriented explanation.
- Do NOT quote legal definitions.
- Explain in practical terms suitable for M&A or industry analysis.
- Return as a dictionary mapping IPC → description.

12. ipc_short_description
- For EACH IPC code,
  provide a VERY SHORT summary (within 7 Korean characters).
- No legal wording.
- Practical and intuitive.
- Return as a dictionary mapping IPC → short label.


----------------------------------------
NAICS CANDIDATES (CHOOSE FROM THIS LIST ONLY)
----------------------------------------
{naics_context}

----------------------------------------
OCR TEXT
----------------------------------------
{text}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You extract patent metadata and select the most appropriate NAICS codes from provided candidates."
            },
            {
                "role": "user",
                "content": prompt
            },
        ],
        temperature=0,
        max_tokens=3000,
    )

    content = response.choices[0].message.content.strip()

    # JSON 안전 추출
    start = content.find("{")
    end = content.rfind("}") + 1
    if start == -1 or end == -1:
        raise ValueError("No valid JSON object found in model response")

    return json.loads(content[start:end])


### 1. input -> user_id : 번호, user_ocr : OCR full text
### output -> 기본 정보 파싱해서 return

In [37]:
user_id = 'yunseo'
user_ocr = """
--- Page 1 (ocr) ---
US 20240139302A1

a2) Patent Application Publication co) Pub. No.: US 2024/0139302 Al

as) United States

Selak et al.

(43) Pub. Date:          May 2, 2024

 

(54) PROPIONIBACTERIUM ACNES
PROPHYLACTIC AND THERAPEUTIC
IMMUNE TREATMENT

(71) Applicant: Origimm Biotechnology GmbH, Wien
(AT)

(72) Inventors: Sanja Selak, Wein (AT); Christine
Triska, Korneuburg (AT); Manfired
Schuster, Schrick (AT); Johannes
Séllner, Wien (AT); Bernhard
Roppenser, Wien (AT); Theresa
Weinhaupl, Wien (AT); Max Réssler,
Wien (AT)

(21) Appl. No.: — 17/801,099

(22) PCT Filed:      Feb. 22, 2021

 

 

 

 

 

 

(86) PCT No.:      PCT/EP2021/054346
8 371 (0001),
(2) Date:       Aug. 19, 2022
(30)            Foreign Application Priority Data
Feb. 21, 2020 (EP) wee cteeeeteneeeees 20158656.7
Feb. 21, 2020 (EP) ...             ... 20158659.1
Feb. 21, 2020 (EP) ...             ... 20158661.7
Feb. 21, 2020 (EP) .[시니늬늬니니이에에에에에아아아 20158662.5
100,000> (> IA1 (NCTC737)
| ES [A2 (P.aen31}
| ES] 1B (KPA171202)
| RSI IC (PV68)
| a tl (HLOSOPA2)
80,000- MHI {Asn12)
은
근 는 50.000-
ey
so
oy       |
o         |
sc
5 = 40,0004
®         |
20,000-
o-           fey ep

oh. oi dn

Publication Classification

(51) Int. CL
AGIK 39/05                 (2006.01)
AGIP 17/10                 (2006.01)
AGIP 31/04                 (2006.01)
CO7K 14/195             (2006.01)
(52) U.S. Cl
CPC .……………   A61K 39/05 (2013.01); A6IP 17/10
(2018.01); A61P 31/04 (2018.01); CO7K
14/195 (2013.01)
(57)                          ABSTRACT

The present invention discloses a vaccine comprising one or
more of Dermatan sulfate-binding adhesin 1 of P. acnes
(DsA1 polypeptide), Dermatan sulfate-binding adhesin 2 of
P. acnes (DsA2 polypeptide), and putative iron-transport
protein (PITP) polypeptide of P acnes, and/or a fragment
and/or derivative of DsAl and/or DsA2 and/or PITP,
wherein the DsAl polypeptide and the DsA2 polypeptide
comprise from N- to C-terminus an N-terminal swapping
region (“NSR”), a first conserved sub-domain (“CSD1”), a
first swapping region (“SR1”), a second conserved sub-
domain (“CSD2”), a second swapping region (“SR2”), a
third conserved sub-domain (“CSD3”), a Pro-Thr repeat
containing region (“PT repeat region”), and a C-terminal
region (“CTR”), and wherein the PITP polypeptide com-
prises from N- to C-terminus an extended neocarzinostatin
family domain (““ENFD”), a first swapping region (“SR1”),
a heme-binding domain (“HbD”), a second swapping region
(“SR2”) including the C-terminal LPXTG motif, and a
hydrophobic C-terminal region (““hLAR”).

Specification includes a Sequence Listing.

    

Soh

 

 

UA SARs

ERAN

 

 

on aon
ee &€ F&F EP SF EF EE S ¥

   

Immunization antigens
"""

In [38]:
def run_NAIC_extract(user_id, user_ocr):
    results = []
    patent_text = user_ocr
    query_vector = embed_patent_text(patent_text)
    naics_candidates = retrieve_top_naics(query_vector, top_k=15)
    naics_context = build_naics_context_text(naics_candidates)
    patent_meta = extract_patent_metadata(
            text=patent_text,
            naics_context=naics_context
        )
    
    results.append({
            "pdf_name": user_id,
            **patent_meta
    })

    return results

In [39]:
text = run_NAIC_extract(user_id, user_ocr)

In [40]:
print(text)

[{'pdf_name': 'yunseo', 'country': 'US', 'title': 'PROPIONIBACTERIUM ACNES PROPHYLACTIC AND THERAPEUTIC IMMUNE TREATMENT', 'applicant_name': 'Origimm Biotechnology GmbH', 'applicant_number': '17/801,099', 'applicant_date': '2021-02-22', 'grant_number': None, 'grant_date': None, 'naics_code': ['325414'], 'abstract': 'The present invention discloses a vaccine comprising one or more of Dermatan sulfate-binding adhesin 1 of P. acnes (DsA1 polypeptide), Dermatan sulfate-binding adhesin 2 of P. acnes (DsA2 polypeptide), and putative iron-transport protein (PITP) polypeptide of P acnes, and/or a fragment and/or derivative of DsAl and/or DsA2 and/or PITP, wherein the DsAl polypeptide and the DsA2 polypeptide comprise from N- to C-terminus an N-terminal swapping region (“NSR”), a first conserved sub-domain (“CSD1”), a first swapping region (“SR1”), a second conserved sub-domain (“CSD2”), a second swapping region (“SR2”), a third conserved sub-domain (“CSD3”), a Pro-Thr repeat containing region 

### temporary -> NAICS_description 파일 한번 정제하기

### 2. naic_df : naic 설명 붙이는 목적

In [42]:
naic_df = pd.read_csv('../data/NAICS_descripition.csv')

def attach_naics_info(results, naic_df):

    naic_map = {
        str(code): {
            "title": title,
            "description": desc
        }
        for code, title, desc in zip(
            naic_df["naics_code"].astype(str),
            naic_df["naics_title"],
            naic_df["description"]
        )
    }

    for item in results:
        codes = item.get("naics_code", [])

        item["naics_info"] = [
            {
                "code": str(code),
                "title": naic_map.get(str(code), {}).get("title"),
                "description": naic_map.get(str(code), {}).get("description")
            }
            for code in codes
        ]

    return results


In [43]:
text = attach_naics_info(text, naic_df)
print(text)

[{'pdf_name': 'yunseo', 'country': 'US', 'title': 'PROPIONIBACTERIUM ACNES PROPHYLACTIC AND THERAPEUTIC IMMUNE TREATMENT', 'applicant_name': 'Origimm Biotechnology GmbH', 'applicant_number': '17/801,099', 'applicant_date': '2021-02-22', 'grant_number': None, 'grant_date': None, 'naics_code': ['325414'], 'abstract': 'The present invention discloses a vaccine comprising one or more of Dermatan sulfate-binding adhesin 1 of P. acnes (DsA1 polypeptide), Dermatan sulfate-binding adhesin 2 of P. acnes (DsA2 polypeptide), and putative iron-transport protein (PITP) polypeptide of P acnes, and/or a fragment and/or derivative of DsAl and/or DsA2 and/or PITP, wherein the DsAl polypeptide and the DsA2 polypeptide comprise from N- to C-terminus an N-terminal swapping region (“NSR”), a first conserved sub-domain (“CSD1”), a first swapping region (“SR1”), a second conserved sub-domain (“CSD2”), a second swapping region (“SR2”), a third conserved sub-domain (“CSD3”), a Pro-Thr repeat containing region 

In [44]:
from pprint import pprint

pprint(text, width=120, sort_dicts=False)


[{'pdf_name': 'yunseo',
  'country': 'US',
  'title': 'PROPIONIBACTERIUM ACNES PROPHYLACTIC AND THERAPEUTIC IMMUNE TREATMENT',
  'applicant_name': 'Origimm Biotechnology GmbH',
  'applicant_number': '17/801,099',
  'applicant_date': '2021-02-22',
  'grant_number': None,
  'grant_date': None,
  'naics_code': ['325414'],
  'abstract': 'The present invention discloses a vaccine comprising one or more of Dermatan sulfate-binding adhesin 1 '
              'of P. acnes (DsA1 polypeptide), Dermatan sulfate-binding adhesin 2 of P. acnes (DsA2 polypeptide), and '
              'putative iron-transport protein (PITP) polypeptide of P acnes, and/or a fragment and/or derivative of '
              'DsAl and/or DsA2 and/or PITP, wherein the DsAl polypeptide and the DsA2 polypeptide comprise from N- to '
              'C-terminus an N-terminal swapping region (“NSR”), a first conserved sub-domain (“CSD1”), a first '
              'swapping region (“SR1”), a second conserved sub-domain (“CSD2”), a sec